# Session 16: Build a Ticket Agent with a Saved Classifier

Reuse one classifier from Sessions 14–15. The saved classifier assigns a category. Qwen decides whether to ask for more information and uses the matching support guide to write a reply.

## 1. Choose a saved classifier

Choose `"distilbert"`, `"gpt2"` or `"mlp"`. First save the classifier in the [Sessions 14–15 practice notebook](https://colab.research.google.com/github/wlos/AI_SS/blob/main/14-15-Sequence-Models-and-Transformers/practice/Session14-15%20Support%20Ticket%20Classification%20Practice.ipynb).

In Colab, select a **T4 GPU** for Qwen and upload the ZIP created by the earlier practice. In a local checkout, you can use the saved folder directly. For the MLP, use a joblib file you saved yourself in an environment with the same scikit-learn version.

In [ ]:
import os
import subprocess
import sys

import torch

CLASSIFIER = "distilbert"
SAVED_MODELS = {
    "distilbert": "saved-ticket-classifier",
    "gpt2": "saved-gpt-classifier",
    "mlp": "saved-count-mlp",
}
model_name = SAVED_MODELS[CLASSIFIER]
archive_name = f"{model_name}.zip"
folders = (
    model_name, f"transformer/{model_name}", f"../transformer/{model_name}",
    f"../../transformer/{model_name}",
)
model_folder = next((folder for folder in folders if os.path.isdir(folder)), None)
if model_folder is None and not os.path.isfile(archive_name):
    raise FileNotFoundError(f"Upload {archive_name} from the Sessions 14–15 practice.")

packages = ["transformers==4.57.1"]
if CLASSIFIER == "mlp":
    packages.extend(["scikit-learn", "joblib"])
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Classifier: {CLASSIFIER} | Qwen device: {device}")

## 2. Load the classifier

The saved DistilBERT and GPT-2 folders include the model, its label mapping and the tokenizer. The MLP bundle includes its word-count vectoriser, the fitted classifier and the label names. All three variants return one category to the agent. The classifier runs on CPU so Qwen can use the GPU.

In [ ]:
from tempfile import TemporaryDirectory
from zipfile import ZipFile

from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline

with TemporaryDirectory() as unpacked:
    if model_folder is None:
        with ZipFile(archive_name) as archive:
            # Read the model and tokenizer files used for inference.
            for filename in (
                "classifier.joblib", "config.json", "model.safetensors",
                "tokenizer.json", "tokenizer_config.json", "special_tokens_map.json",
                "vocab.json", "vocab.txt", "merges.txt", "added_tokens.json",
            ):
                if filename in archive.namelist():
                    archive.extract(filename, unpacked)
        source = unpacked
    else:
        source = model_folder

    if CLASSIFIER == "mlp":
        import joblib

        saved_mlp = joblib.load(f"{source}/classifier.joblib")
    else:
        classifier = pipeline(
            "text-classification",
            model=AutoModelForSequenceClassification.from_pretrained(
                source, local_files_only=True, use_safetensors=True,
            ),
            tokenizer=AutoTokenizer.from_pretrained(source, local_files_only=True),
            device=-1,
        )


def classify_ticket(ticket):
    if CLASSIFIER == "mlp":
        counts = saved_mlp["vectorizer"].transform([ticket])
        prediction = saved_mlp["classifier"].predict(counts)[0]
        category = saved_mlp["class_names"][int(prediction)]
    else:
        result = classifier(ticket, truncation=True, max_length=96)
        category = result[0]["label"]
    return category if category in CATEGORIES else None

## 3. Load the support guides

Read the same local JSON guides as the adapter version of the ticket agent.

In [ ]:
import json
from urllib.request import urlretrieve

SESSION_DIR = "16-LLM-Fine-Tuning-and-Agents"
GUIDE_FILE = "sample_data/support_guides.json"
guide_paths = ("support_guides.json", GUIDE_FILE, f"../{GUIDE_FILE}", f"{SESSION_DIR}/{GUIDE_FILE}")
guide_path = next((path for path in guide_paths if os.path.isfile(path)), None)
if guide_path is None:
    guide_url = f"https://raw.githubusercontent.com/wlos/AI_SS/main/{SESSION_DIR}/{GUIDE_FILE}"
    guide_path, _ = urlretrieve(guide_url, "support_guides.json")

with open(guide_path, encoding="utf-8") as guide_file:
    support_guides = json.load(guide_file)

CATEGORIES = ("Account access", "Hardware", "Network", "Software")
print("Available guides:", ", ".join(support_guides))

## 4. Load Qwen for decisions and replies

Load the original Qwen instruct checkpoint. The separate classifier handles the category prediction.

In [ ]:
from transformers import AutoModelForCausalLM, GenerationConfig

CHECKPOINT = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)
dtype = torch.float16 if device.type == "cuda" else torch.float32
model = AutoModelForCausalLM.from_pretrained(CHECKPOINT, dtype=dtype).to(device)
model.eval()


@torch.inference_mode()
def generate_reply(prompt, max_new_tokens=64):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    ).to(device)
    generation = GenerationConfig(
        do_sample=False,
        max_new_tokens=max_new_tokens,
        eos_token_id=model.generation_config.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )
    tokens = model.generate(**inputs, generation_config=generation)
    reply = tokens[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(reply, skip_special_tokens=True).strip()

## 5. Run the agent

Qwen chooses whether to ask for clarification or proceed. Python calls the saved classifier and looks up the category in the guides. Qwen uses the guide result to draft the reply.

In [ ]:
def run_ticket_agent(ticket, guides):
    decision_prompt = (
        "Decide whether the support ticket has enough information to proceed. "
        "Return exactly ASK or CLASSIFY. "
        "For a vague login issue without the service name, return ASK. "
        "For a specific account, device, network or software issue, return CLASSIFY.\n"
        f"Ticket: {ticket}"
    )
    decision = generate_reply(decision_prompt, max_new_tokens=8).upper()
    needs_clarification = decision != "CLASSIFY"
    trace = {
        "decision": decision,
        "category": None,
        "guide": None,
        "needs_clarification": needs_clarification,
    }
    if needs_clarification:
        trace["reply"] = "Which service or device is affected, and what happened?"
        return trace

    category = classify_ticket(ticket)
    trace["category"] = category
    if category is None:
        trace["reply"] = "I could not classify this ticket. Please contact the service desk."
        return trace

    guide = guides.get(category)
    trace["guide"] = guide
    if guide is None:
        trace["reply"] = "I could not find a guide for this issue. Please contact the service desk."
        return trace

    trace["reply"] = generate_reply(
        f"Write a short support reply using only the guide below.\nTicket: {ticket}\nGuide: {guide}"
    )
    return trace

## 6. Chat with the ticket agent

Try **I can't log in.** If the agent asks for details, reply **My student portal account is locked.** The chat combines both messages. After a final reply, the next message starts a new ticket.

Type `/exit` to stop. Inspect `last_trace` to see the decision, category and guide.

In [ ]:
ticket_context = []
last_trace = None
print("Ticket support chat. Type /exit to stop.")

while True:
    message = input("You: ").strip()
    if message.casefold() == "/exit":
        break
    if not message:
        continue

    ticket_context.append(message)
    last_trace = run_ticket_agent(" ".join(ticket_context), support_guides)
    print(f"Agent: {last_trace['reply']}")

    # Keep earlier details while the agent is waiting for clarification.
    if not last_trace["needs_clarification"]:
        ticket_context.clear()